# Phase 07 — Graph Intelligence Layer
## Sentinel AI: Enterprise Fraud Intelligence Platform

---

### Graph Philosophy

Traditional ML treats each transaction as an **isolated row**.
Graph Intelligence treats each transaction as a **node in a network**.

| Paradigm | Question |
|---|---|
| Traditional ML | *Is this transaction fraudulent?* |
| Graph Intelligence | *Which entities, communities, and relationships around this transaction increase or decrease its risk?* |

Banks do not investigate transactions in isolation. They investigate **networks**.

### Dataset Transparency
The Bank of India dataset is strictly anonymized (features `F1...F4000`). It does not contain explicit entity identifiers.
To demonstrate enterprise graph intelligence without fabricating scientific claims, this notebook uses a **Deterministic Synthetic Topology Layer**.
Every transaction is mapped to synthetic Customers, Devices, IPs, and Merchants using SHA-256 hashing — guaranteeing reproducibility and zero randomness.

### Notebook Role
This notebook is a **thin orchestrator**. All logic lives in reusable modules under `src/graph/`.

In [1]:
from src.utils.bootstrap import bootstrap
bootstrap()

from pathlib import Path
import sys

import json
import sys
from pathlib import Path

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Ensure project root is on the path

from src.graph.GraphBuilder import GraphBuilder
from src.graph.CommunityDetector import CommunityDetector
from src.graph.CentralityEngine import CentralityEngine
from src.graph.RiskPropagationEngine import RiskPropagationEngine
from src.graph.GraphExporter import GraphExporter
from src.graph.GraphFeatureStore import GraphFeatureStore

# Output directories
REPORTS_DIR = "reports/phase_07"
VIZ_DIR = "reports/phase_07/visualization"
Path(REPORTS_DIR).mkdir(parents=True, exist_ok=True)
Path(VIZ_DIR).mkdir(parents=True, exist_ok=True)

print("Sentinel AI Graph Intelligence Engine initialised.")

Sentinel AI Graph Intelligence Engine initialised.


## Section 1 — Data Ingestion

In [2]:
df = pd.read_csv("data/selected/approved_features.csv")

TARGET = "F3924"
fraud_labels = df[TARGET]
features = df.drop(columns=[TARGET])

print(f"Transactions: {len(features):,}")
print(f"Fraud rate: {fraud_labels.mean()*100:.2f}%")
print(f"Features: {features.shape[1]}")

Transactions: 9,082
Fraud rate: 0.89%
Features: 535


## Section 2 — Graph Schema & Construction
The `GraphBuilder` deterministically maps each transaction to synthetic entities using SHA-256.
See `docs/GRAPH_SCHEMA.md` for the full node/edge type definitions.

In [3]:
builder = GraphBuilder()
G = builder.build(features, fraud_labels)
metadata = builder.get_metadata()
builder.save_metadata(REPORTS_DIR)

print("=== Graph Metadata ===")
for k, v in metadata.items():
    print(f"  {k}: {v}")

=== Graph Metadata ===
  graph_version: 1.0
  builder_version: DeterministicSyntheticBuilder v1.0
  build_strategy: SHA-256 deterministic hashing with modular entity pools
  seed: sentinel_ai
  dataset_hash: fd7e474be14cc9dd
  build_timestamp: 2026-06-30T21:12:40Z
  execution_time_seconds: 0.251
  total_nodes: 16027
  total_edges: 36258
  node_type_counts: {'Transaction': 9082, 'Customer': 4167, 'Device': 1978, 'IP_Address': 300, 'Merchant': 500}
  edge_type_counts: {'SENT_TO': 9082, 'EXECUTED_ON': 9082, 'INITIATED': 9082, 'CONNECTED_VIA': 9012}
  pool_sizes: {'customers': 5000, 'devices': 2000, 'merchants': 500, 'ips': 300}


## Section 3 — Graph Statistics & Quality Metrics

In [4]:
centrality_engine = CentralityEngine()

# Basic network stats
G_und = G.to_undirected()
network_stats = {
    "total_nodes": G.number_of_nodes(),
    "total_edges": G.number_of_edges(),
    "density": round(nx.density(G_und), 6),
    "connected_components": nx.number_connected_components(G_und),
    "average_degree": round(sum(dict(G_und.degree()).values()) / G_und.number_of_nodes(), 4),
}

# Advanced quality metrics
quality_metrics = centrality_engine.compute_graph_quality_metrics(G)
network_stats.update(quality_metrics)

with open(Path(REPORTS_DIR) / "network_statistics.json", "w") as f:
    json.dump(network_stats, f, indent=2)

print("=== Network Statistics ===")
for k, v in network_stats.items():
    print(f"  {k}: {v}")

=== Network Statistics ===
  total_nodes: 16027
  total_edges: 36258
  density: 0.000282
  connected_components: 1
  average_degree: 4.5246
  assortativity: -0.00051
  average_clustering_coefficient: 0.0
  transitivity: 0
  triangle_count: 0
  average_neighbor_degree: 9.4261


## Section 4 — Centrality Analysis
Computing PageRank, Betweenness, Degree, Closeness, and Eigenvector centrality for every node.

In [5]:
centrality_scores = centrality_engine.compute_all(G)
centrality_records = centrality_engine.to_records(centrality_scores)

centrality_df = pd.DataFrame(centrality_records)
centrality_df.to_csv(Path(REPORTS_DIR) / "centrality_scores.csv", index=False)

print(f"Centrality scores computed for {len(centrality_df):,} nodes.")
print("\nTop 10 by PageRank:")
print(centrality_df.nlargest(10, 'pagerank')[['node_id', 'pagerank', 'degree', 'betweenness']].to_string(index=False))

Centrality scores computed for 16,027 nodes.

Top 10 by PageRank:
 node_id  pagerank   degree  betweenness
IP_00088  0.000886 0.002870     0.010393
IP_00100  0.000864 0.002808     0.007751
IP_00024  0.000860 0.002808     0.006843
IP_00170  0.000821 0.002621     0.008233
IP_00291  0.000818 0.002621     0.010524
IP_00089  0.000814 0.002621     0.008192
IP_00090  0.000806 0.002558     0.007487
IP_00050  0.000805 0.002558     0.010125
IP_00078  0.000804 0.002496     0.005475
IP_00105  0.000792 0.002558     0.009314


## Section 5 — Community Detection
Identifying transaction communities using Greedy Modularity optimisation.

In [6]:
detector = CommunityDetector()
community_map = detector.detect(G, method="greedy")

community_registry = detector.build_registry(G, community_map)
suspicious_communities = detector.get_suspicious_communities(community_registry)

pd.DataFrame(community_registry).to_csv(Path(REPORTS_DIR) / "community_registry.csv", index=False)

print(f"Communities detected: {len(community_registry)}")
print(f"Suspicious clusters: {len(suspicious_communities)}")
print("\nCommunity Registry (Top 10 by size):")
print(pd.DataFrame(community_registry).nlargest(10, 'size')[['community_id', 'size', 'fraud_percentage', 'density', 'risk_tier']].to_string(index=False))

Communities detected: 36
Suspicious clusters: 0

Community Registry (Top 10 by size):
 community_id  size  fraud_percentage  density risk_tier
            0  1536            0.0084 0.001902       Low
            1  1494            0.0061 0.001928       Low
            2  1384            0.0065 0.001923       Low
            3  1290            0.0111 0.002046       Low
            4  1225            0.0073 0.002140       Low
            5   949            0.0019 0.002499       Low
            6   813            0.0044 0.002933       Low
            7   678            0.0075 0.003355       Low
            8   636            0.0083 0.003531       Low
            9   624            0.0083 0.003648       Low


## Section 6 — Iterative Risk Propagation
Fraud risk flows from known-fraud Transactions outward through shared Devices, IPs, and Customers using iterative message passing (damping=0.85) until convergence.

In [7]:
risk_engine = RiskPropagationEngine(damping=0.85, max_iterations=10, epsilon=0.001)
risk_scores = risk_engine.propagate(G)
community_risks = risk_engine.get_community_risks(community_map)

entity_registry = risk_engine.build_entity_registry(G, community_map)
pd.DataFrame(entity_registry).to_csv(Path(REPORTS_DIR) / "entity_registry.csv", index=False)

print(f"Risk propagation converged in {risk_engine.iterations_run} iterations.")
print(f"Entity registry: {len(entity_registry):,} entities")

# Risk tier distribution
tier_dist = pd.DataFrame(entity_registry)['risk_tier'].value_counts()
print("\nEntity Risk Tier Distribution:")
print(tier_dist.to_string())

Risk propagation converged in 7 iterations.
Entity registry: 6,945 entities

Entity Risk Tier Distribution:
risk_tier
Low         6875
Elevated      56
Critical      12
High           2


## Section 7 — Suspicious Community Deep Dive

In [8]:
if suspicious_communities:
    print(f"\n=== {len(suspicious_communities)} Suspicious Communities ===")
    for sc in suspicious_communities:
        print(f"\nCommunity {sc['community_id']}:")
        print(f"  Size: {sc['size']}")
        print(f"  Fraud %: {sc['fraud_percentage']*100:.1f}%")
        print(f"  Density: {sc['density']:.4f}")
        print(f"  Risk Tier: {sc['risk_tier']}")
        print(f"  Central Node: {sc['central_node']}")
else:
    print("No communities met the suspicion threshold (fraud% > 30% AND above-average density).")
    print("This is expected when the overall fraud rate is low.")
    print("\nTop communities by fraud percentage:")
    top_fraud = sorted(community_registry, key=lambda x: x['fraud_percentage'], reverse=True)[:5]
    for c in top_fraud:
        print(f"  Community {c['community_id']}: {c['fraud_percentage']*100:.1f}% fraud, size={c['size']}")

No communities met the suspicion threshold (fraud% > 30% AND above-average density).
This is expected when the overall fraud rate is low.

Top communities by fraud percentage:
  Community 35: 5.0% fraud, size=69
  Community 34: 4.5% fraud, size=72
  Community 32: 4.1% fraud, size=82
  Community 22: 2.7% fraud, size=201
  Community 14: 2.4% fraud, size=349


## Section 8 — Graph Feature Store
Assembling all graph-derived features into a single Parquet feature store for Phase 8 GNN consumption.

In [9]:
feature_store = GraphFeatureStore()
store_df = feature_store.build(G, centrality_scores, community_map, community_risks, risk_scores)
feature_store.save(store_df, REPORTS_DIR)

print(f"Graph Feature Store: {store_df.shape[0]:,} rows x {store_df.shape[1]} columns")
print(f"\nFeature summary:")
print(store_df.describe().round(4).to_string())

Graph Feature Store: 16,027 rows x 13 columns

Feature summary:
           degree    pagerank  betweenness   closeness  eigenvector  community_id  community_size  community_risk  entity_risk  neighbor_fraud_ratio  graph_distance_to_fraud
count  16027.0000  16027.0000   16027.0000  16027.0000   16027.0000    16027.0000      16027.0000      16027.0000   16027.0000            16027.0000                16027.000
mean       0.0003      0.0001       0.0003      0.1730       0.0034        8.6989        866.9037          0.0088       0.0088                0.0032                    3.654
std        0.0003      0.0001       0.0009      0.0180       0.0071        8.4813        510.5320          0.0068       0.0806                0.0387                    0.995
min        0.0001      0.0000       0.0000      0.1388       0.0000        0.0000         69.0000          0.0004       0.0000                0.0000                    0.000
25%        0.0002      0.0000       0.0000      0.1641       0.000

## Section 9 — Visualization Export
Exporting the enriched graph to GEXF (Gephi), GraphML, and React Flow / Cytoscape JSON.

In [10]:
exporter = GraphExporter()
exporter.export_all(
    graph=G,
    output_dir=VIZ_DIR,
    centrality_scores=centrality_scores,
    community_map=community_map,
    risk_scores=risk_scores,
)

print("Exported:")
for f in Path(VIZ_DIR).iterdir():
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:.1f} KB")

Exported:
  edges.csv: 1150.7 KB
  edges.json: 3593.8 KB
  graph.gexf: 10720.7 KB
  graph.graphml: 6578.2 KB
  nodes.csv: 619.8 KB
  nodes.json: 2703.7 KB


## Section 10 — Graph Explainability Sample
Demonstrating how the Sentinel Investigator can surface network context for individual cases.

In [11]:
# Pick a sample transaction
sample_txn = "TXN_000000"
G_und = G.to_undirected()

neighbours = list(G_und.neighbors(sample_txn))
txn_community = community_map.get(sample_txn, -1)
txn_risk = risk_scores.get(sample_txn, 0.0)
txn_pagerank = centrality_scores.get(sample_txn, {}).get('pagerank', 0.0)

# Count fraud in neighbourhood
fraud_neighbors = sum(1 for n in neighbours if G.nodes[n].get('fraud', 0) == 1)

print(f"=== Network Context for {sample_txn} ===")
print(f"  Community: {txn_community}")
print(f"  Community Risk: {community_risks.get(txn_community, 0.0)*100:.1f}%")
print(f"  PageRank: {txn_pagerank:.6f}")
print(f"  Connected Entities: {len(neighbours)}")
print(f"  Connected Fraud Nodes: {fraud_neighbors}")
print(f"  Entity Risk: {txn_risk*100:.1f}%")
print(f"\n  Connected entities:")
for n in neighbours:
    print(f"    {n} ({G.nodes[n].get('type')}) — risk: {risk_scores.get(n, 0)*100:.1f}%")

=== Network Context for TXN_000000 ===
  Community: 1
  Community Risk: 0.6%
  PageRank: 0.000051
  Connected Entities: 3
  Connected Fraud Nodes: 0
  Entity Risk: 0.0%

  Connected entities:
    MER_00490 (Merchant) — risk: 0.0%
    DEV_00018 (Device) — risk: 0.4%
    CUS_04025 (Customer) — risk: 0.0%


## Section 11 — Phase 7 Summary

### Artifacts Generated
| Artifact | Location | Purpose |
|---|---|---|
| `graph_metadata.json` | `reports/phase_07/` | Graph versioning and build provenance |
| `network_statistics.json` | `reports/phase_07/` | Topology and quality metrics |
| `centrality_scores.csv` | `reports/phase_07/` | Node-level centrality features |
| `community_registry.csv` | `reports/phase_07/` | Community-level statistics and risk tiers |
| `entity_registry.csv` | `reports/phase_07/` | Entity-level risk scores and tiers |
| `graph_feature_store.parquet` | `reports/phase_07/` | Phase 8 GNN input features |
| `graph.gexf` | `reports/phase_07/visualization/` | Gephi import |
| `graph.graphml` | `reports/phase_07/visualization/` | Standard graph exchange |
| `nodes.json` / `edges.json` | `reports/phase_07/visualization/` | React Flow / Cytoscape frontend |

### Key Insight
Sentinel AI no longer thinks: *"Is this transaction fraudulent?"*

It now thinks: *"Which entities, communities, and relationships around this transaction increase or decrease its risk?"*

This graph intelligence layer is the foundation for Phase 8 (Graph Neural Networks).